### Allscripts Sunrise (SCM) - Measurement Hydration

**Source Tables:**
- _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur (observations with numeric values)
- _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence (links to patients)

**Strategy:**
- Same source as observations, but filter for numeric values
- Parse ValueText as numeric where possible
- Map ObsItemGUID to OMOP measurement concepts
- Use UnitOfMeasure for measurement units

**Note:**
- Sunrise stores both observations and measurements in cv3observationcur
- Distinguish by attempting numeric conversion of ValueText

In [ ]:
source = 'allscripts_scm'

# Transformation

In [ ]:
silver_measurement_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(meas_concept.omop_concept_id, 0) AS measurement_concept_id,
  CAST(obs.ETL_LOAD_TS AS DATE) AS measurement_date,
  obs.ETL_LOAD_TS AS measurement_datetime,
  NULL AS measurement_time,
  44818702 AS measurement_type_concept_id,  -- EHR
  NULL AS operator_concept_id,
  CAST(obs.ValueText AS DOUBLE) AS value_as_number,
  NULL AS value_as_concept_id,
  0 AS unit_concept_id,
  NULL AS range_low,
  NULL AS range_high,
  NULL AS provider_id,
  NULL AS visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT('{source}', ' | ', obs.GUID) AS measurement_source_value,
  0 AS measurement_source_concept_id,
  obs.UnitOfMeasure AS unit_source_value,
  obs.ValueText AS value_source_value,
  '{source}' AS source_system
FROM _exponent._bronze_allscripts_scm_prod01_vw.dbo_cv3observationcur obs
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3ordertaskoccurrence oto
  ON obs.OrderTaskOccuranceGUID = oto.GUID
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', CHAR(31), 'dbo_cv3client', CHAR(31), 'guid', CHAR(31), CAST(oto.ClientGUID AS BIGINT)) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
  ON meas_concept.source_id = CAST(obs.ObsItemGUID AS STRING)
  AND meas_concept.domain_id = 'Measurement'
  AND meas_concept.source_system = '{source}'
WHERE obs.GUID IS NOT NULL
  AND oto.ClientGUID IS NOT NULL
  AND obs.StatusType = 1  -- Performed
  AND obs.ValueText IS NOT NULL
  AND obs.ValueText RLIKE '^[0-9]+\\.?[0-9]*$'  -- Numeric values only
LIMIT 10000
''')

display(silver_measurement_df)
silver_measurement_df.createOrReplaceTempView("silver_measurement")

# Merge to Silver

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.measurement AS t
USING (
  SELECT * FROM silver_measurement 
  WHERE measurement_date IS NOT NULL
) AS s
ON t.measurement_source_value = s.measurement_source_value

WHEN MATCHED THEN UPDATE SET
  t.person_id = s.person_id,
  t.measurement_concept_id = s.measurement_concept_id,
  t.measurement_date = s.measurement_date,
  t.measurement_datetime = s.measurement_datetime,
  t.measurement_time = s.measurement_time,
  t.measurement_type_concept_id = s.measurement_type_concept_id,
  t.operator_concept_id = s.operator_concept_id,
  t.value_as_number = s.value_as_number,
  t.value_as_concept_id = s.value_as_concept_id,
  t.unit_concept_id = s.unit_concept_id,
  t.range_low = s.range_low,
  t.range_high = s.range_high,
  t.provider_id = s.provider_id,
  t.visit_occurrence_id = s.visit_occurrence_id,
  t.visit_detail_id = s.visit_detail_id,
  t.measurement_source_concept_id = s.measurement_source_concept_id,
  t.unit_source_value = s.unit_source_value,
  t.value_source_value = s.value_source_value

WHEN NOT MATCHED THEN INSERT (
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  value_source_value,
  source_system
)
VALUES (
  s.person_id,
  s.measurement_concept_id,
  s.measurement_date,
  s.measurement_datetime,
  s.measurement_time,
  s.measurement_type_concept_id,
  s.operator_concept_id,
  s.value_as_number,
  s.value_as_concept_id,
  s.unit_concept_id,
  s.range_low,
  s.range_high,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.measurement_source_value,
  s.measurement_source_concept_id,
  s.unit_source_value,
  s.value_source_value,
  s.source_system
);

# Populate Mapping Table

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_measurement (
  measurement_source_value,
  active_flag,
  created_at,
  updated_at
)
SELECT 
  measurement_source_value,
  TRUE,
  CURRENT_TIMESTAMP(),
  CURRENT_TIMESTAMP()
FROM _exponent.omop_silver.measurement
WHERE measurement_source_value NOT IN (
  SELECT measurement_source_value 
  FROM _exponent.omop_mapping.source_to_measurement
  WHERE active_flag = TRUE
);

# Merge to Gold

In [ ]:
%sql
MERGE INTO _exponent.omop.measurement AS gold
USING (
  SELECT 
    source_to_measurement.measurement_id,
    s.person_id,
    s.measurement_concept_id,
    s.measurement_date,
    s.measurement_datetime,
    s.measurement_time,
    s.measurement_type_concept_id,
    s.operator_concept_id,
    s.value_as_number,
    s.value_as_concept_id,
    s.unit_concept_id,
    s.range_low,
    s.range_high,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.measurement_source_value,
    s.measurement_source_concept_id,
    s.unit_source_value,
    s.value_source_value
  FROM _exponent.omop_silver.measurement s
  JOIN _exponent.omop_mapping.source_to_measurement
    ON source_to_measurement.measurement_source_value = s.measurement_source_value
    AND source_to_measurement.active_flag = TRUE
) AS src
ON gold.measurement_id = src.measurement_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id = src.person_id,
  gold.measurement_concept_id = src.measurement_concept_id,
  gold.measurement_date = src.measurement_date,
  gold.measurement_datetime = src.measurement_datetime,
  gold.measurement_time = src.measurement_time,
  gold.measurement_type_concept_id = src.measurement_type_concept_id,
  gold.operator_concept_id = src.operator_concept_id,
  gold.value_as_number = src.value_as_number,
  gold.value_as_concept_id = src.value_as_concept_id,
  gold.unit_concept_id = src.unit_concept_id,
  gold.range_low = src.range_low,
  gold.range_high = src.range_high,
  gold.provider_id = src.provider_id,
  gold.visit_occurrence_id = src.visit_occurrence_id,
  gold.visit_detail_id = src.visit_detail_id,
  gold.measurement_source_value = src.measurement_source_value,
  gold.measurement_source_concept_id = src.measurement_source_concept_id,
  gold.unit_source_value = src.unit_source_value,
  gold.value_source_value = src.value_source_value

WHEN NOT MATCHED THEN INSERT (
  measurement_id,
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  value_source_value
)
VALUES (
  src.measurement_id,
  src.person_id,
  src.measurement_concept_id,
  src.measurement_date,
  src.measurement_datetime,
  src.measurement_time,
  src.measurement_type_concept_id,
  src.operator_concept_id,
  src.value_as_number,
  src.value_as_concept_id,
  src.unit_concept_id,
  src.range_low,
  src.range_high,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.measurement_source_value,
  src.measurement_source_concept_id,
  src.unit_source_value,
  src.value_source_value
);